In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/db/impact_ia.db')

# Toutes les tables et leur contenu
tables = ['Fournisseur', 'Modele', 'Metrique_Energie', 'Metrique_Qualite', 
          'Metrique_Categorie', 'Benchmark_Tache', 'Tarif', 
          'Facteur_Emission', 'FMTI_Indicateur', 'FMTI_Score']

for t in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t} : {count} lignes')

Fournisseur : 16 lignes
Modele : 116 lignes
Metrique_Energie : 116 lignes
Metrique_Qualite : 116 lignes
Metrique_Categorie : 138866 lignes
Benchmark_Tache : 412 lignes
Tarif : 20 lignes
Facteur_Emission : 149 lignes
FMTI_Indicateur : 100 lignes
FMTI_Score : 1300 lignes


In [2]:
# Les nouveaux modèles ajoutés par Louis
pd.read_sql_query("""
    SELECT mdl_nom, frs_nom, mdl_nb_params
    FROM Modele m
    LEFT JOIN Fournisseur f ON m.frs_id = f.frs_id
    ORDER BY frs_nom, mdl_nom
""", conn)

,mdl_nom,frs_nom,mdl_nb_params
0,EuroLLM-22B-Instruct-2512,None,NaN
1,Yi-1.5-9B-Chat,None,NaN
2,chocolatine-14b-instruct-dpo-v1.2-q4,None,14.0
3,chocolatine-2-14b-instruct-v2.0.3-q8,None,7.0
4,glm-4.5,None,355.0
...,...,...,...
111,o4-mini,OpenAI,18.0
112,grok-3-mini-beta,xAI,180.0
113,grok-4-fast,xAI,1700.0
114,grok-4.1-fast,xAI,1700.0


In [3]:
# Le benchmark par tâche
pd.read_sql_query("""
    SELECT m.mdl_nom, b.btk_tache, b.btk_gpu_modele, 
           b.btk_joules_par_token, b.btk_kwh_par_1k_tokens
    FROM Benchmark_Tache b
    LEFT JOIN Modele m ON b.mdl_id = m.mdl_id
    ORDER BY b.btk_kwh_par_1k_tokens ASC
    LIMIT 20
""", conn)

,mdl_nom,btk_tache,btk_gpu_modele,btk_joules_par_token,btk_kwh_par_1k_tokens
0,gpt-oss-20b,gpqa,B200,0.028447,0.000008
1,gpt-oss-20b,gpqa,B200,0.029096,0.000008
2,gpt-oss-20b,gpqa,B200,0.031752,0.000009
3,gpt-oss-20b,gpqa,B200,0.039304,0.000011
4,gpt-oss-120b,gpqa,B200,0.039667,0.000011
5,gpt-oss-20b,gpqa,B200,0.040652,0.000011
6,gpt-oss-120b,gpqa,B200,0.042069,0.000012
7,gpt-oss-120b,gpqa,B200,0.043849,0.000012
8,gpt-oss-20b,gpqa,H100,0.044216,0.000012
9,gpt-oss-20b,gpqa,H100,0.044664,0.000012


In [4]:
# Le détecteur de gaspillage
pd.read_sql_query("""
    SELECT m.mdl_nom, f.frs_nom,
           e.nrg_wh_par_token,
           AVG(b.btk_kwh_par_1k_tokens) as kwh_labo
    FROM Modele m
    LEFT JOIN Fournisseur f ON m.frs_id = f.frs_id
    LEFT JOIN Metrique_Energie e ON m.mdl_id = e.mdl_id
    LEFT JOIN Benchmark_Tache b ON m.mdl_id = b.mdl_id
    WHERE e.nrg_wh_par_token IS NOT NULL 
    AND b.btk_kwh_par_1k_tokens IS NOT NULL
    GROUP BY m.mdl_id
    ORDER BY m.mdl_nom
""", conn)

,mdl_nom,frs_nom,nrg_wh_par_token,kwh_labo
0,Qwen3-Coder-480B-A35B-Instruct,Alibaba,0.001951,0.002610
1,deepseek-r1-0528,DeepSeek,0.003979,0.001586
2,deepseek-v3-chat,DeepSeek,0.003979,0.000957
3,gemma-3-12b,Google,0.000094,0.000088
4,gemma-3-27b,Google,0.000112,0.000188
5,gpt-oss-120b,OpenAI,0.000342,0.000065
6,gpt-oss-20b,OpenAI,0.000083,0.000026
7,llama-3.1-405b,Meta,0.009134,0.001296
8,llama-3.1-70b,Meta,0.000658,0.000261
9,llama-3.1-8b,Meta,0.000089,0.000039
